In [1]:
import json

In [2]:
with open("../../data/retacred/test.json") as file:
  data = json.load(file)

In [3]:
# get the sequence lengths
seq_lens = []
for d in data:
  if d['subj_end'] < d['obj_end']:
    seq_lens.append((d['id'], d['obj_end'] - d['subj_start']))
  else:
    seq_lens.append((d['id'], d['subj_end'] - d['obj_start']))
  # seq_lens.append((d['id'], len(d['token'])))
  
seq_lens = sorted(list(set(seq_lens)), key=lambda x: x[1], reverse=False)
print(len(seq_lens))
print(seq_lens[0])

13418
('098f689f60db3293b13b', 4)


In [4]:
# bucket mapping function
def find_bucket_id(token_count):
  if token_count <= 10:
      return 0
  elif token_count > 36:
      return (36 - 11) // 5 + 1
  else:
      tmp_cnt = token_count - 11
      return tmp_cnt // 5 + 1

0.15002235802653152
0.7000298107020421
0.14994783127142644


In [5]:
num_docs = len(seq_lens)

buckets = dict()

# get the IDs
for i, (did, dtok) in enumerate(seq_lens):
    buckets[find_bucket_id(dtok)] = buckets.get(find_bucket_id(dtok), []) + [did]

# print the size of buckets
for k, v in buckets.items():
    print(f"bucket {k}: {len(v) / num_docs * 100:.2f}%")

num_buckets = len(buckets)
print(num_buckets)


2013
9393
2012


In [6]:
# create buckets
new_tests = dict()

for d in data:
    for k, v in buckets.items():
        if d['id'] in v:
            new_tests[k] = new_tests.get(k, []) + [d]
            break

for k, v in new_tests.items():
    print(f"bucket {k}: {len(v)}")

In [7]:
# write the files
for k, v in new_tests.items():
    with open(f"../../data/retacred/test_bucket_{k}.json", "w") as file:
        json.dump(v, file)

c:\Users\Alex\Desktop\Text Mining Coursework\COMP61332-re\src_ra_cgcn


'c:\\Users\\Alex\\Desktop\\Text Mining Coursework\\COMP61332-re\\src_ra_cgcn'

In [ ]:
# CD into the parent folder for convinience

%cd ../../src_ra_cgcn
%pwd


In [9]:
# imports

import random
import argparse

from tqdm import tqdm
import torch

from data.loader import DataLoader, DataLoaderPredict
from model.trainer import GCNTrainer
from utils import torch_utils, scorer, constant, helper
from utils.vocab import Vocab


In [10]:
def load_model(model_dir, model="best_model.pt", seed=1234, cuda=None):
    # set cuda and cuda seed
    if cuda is None:
        cuda = torch.cuda.is_available()
        torch.cuda.manual_seed(seed)

    # set the seeds
    torch.manual_seed(seed)
    random.seed(seed)

    # load opt
    model_file = f"{model_dir}/{model}"
    print(f"Loading model from {model_file}")
    opt = torch_utils.load_config(model_file)
    trainer = GCNTrainer(opt)
    trainer.load(model_file)

    # load vocab
    vocab_file = f"{model_dir}/vocab.pkl"
    vocab = Vocab(vocab_file, load=True)
    assert opt['vocab_size'] == vocab.size, "Vocab size must match that in the saved model."

    return trainer, vocab, opt

In [11]:
def evaluate_model(trainer, vocab, data_dir="../data/retacred", opt=None, scorer=None, test_file="test.json"):
    assert opt is not None, "opt must not be None."
    assert scorer is not None, "scorer must not be None."

    # load data
    data_file = f"{data_dir}/{test_file}"
    print(f"Loading data from {data_file} with batch size {opt['batch_size']}...")
    batch = DataLoader(data_file, opt['batch_size'], opt, vocab, evaluation=True)

    label2id = constant.LABEL_TO_ID
    id2label = dict([(v,k) for k,v in label2id.items()])

    predictions = []
    all_probs = []
    batch_iter = tqdm(batch)
    for i, b in enumerate(batch_iter):
        preds, probs, _ = trainer.predict(b)
        predictions += preds
        all_probs += probs

    predictions = [id2label[p] for p in predictions]
    _, details = scorer.score(batch.gold(), predictions, verbose=False)

    # print("Detailed evaluation:")
    # print("\n".join([f"{k}: {v}" for k, v in details.items()]))
    print("Evaluation ended.")

Loading model from saved_models/400//best_model.pt


c:\Users\Alex\Desktop\Text Mining Coursework\COMP61332-re\src_ra_cgcn\utils\torch_utils.py:158: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dump = torch.load(filename)


Finetune all embeddings.
Vocab size 50115 loaded from file


c:\Users\Alex\Desktop\Text Mining Coursework\COMP61332-re\src_ra_cgcn\model\trainer.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename)


Loading model from saved_models/500//best_model.pt
Finetune all embeddings.
Vocab size 50115 loaded from file


In [12]:
trainer_agcn, vocab_agcn, opt_agcn = load_model(model_dir="saved_models/400/", model="best_model.pt")
trainer_gcn, vocab_gcn, opt_gcn = load_model(model_dir="saved_models/500/", model="best_model.pt")

Loading data from ../data/retacred/test_bottom.json with batch size 50...
41 batches created for ../data/retacred/test_bottom.json


100%|██████████| 41/41 [00:01<00:00, 28.80it/s]


Precision (micro): 77.213%
   Recall (micro): 81.263%
       F1 (micro): 79.186%
Evaluation ended.


In [13]:
# eval AGCN
for k in range(num_buckets):
    evaluate_model(trainer=trainer_agcn, vocab=vocab_agcn, opt=opt_agcn, scorer=scorer, test_file=f"test_bucket_{k}.json")

Loading data from ../data/retacred/test_mid.json with batch size 50...
188 batches created for ../data/retacred/test_mid.json


100%|██████████| 188/188 [00:03<00:00, 48.01it/s]

Precision (micro): 78.899%
   Recall (micro): 76.748%
       F1 (micro): 77.808%
Evaluation ended.


In [14]:
# eval GCN
for k in range(num_buckets):
    evaluate_model(trainer=trainer_gcn, vocab=vocab_gcn, opt=opt_gcn, scorer=scorer, test_file=f"test_bucket_{k}.json")

Loading data from ../data/retacred/test_top.json with batch size 50...
41 batches created for ../data/retacred/test_top.json


100%|██████████| 41/41 [00:01<00:00, 31.92it/s]

Precision (micro): 76.609%
   Recall (micro): 71.765%
       F1 (micro): 74.108%
Evaluation ended.


In [15]:
# plotting 
import matplotlib.pyplot as plt

# data to plot
# att_gcn = [79.082, 77.804, 74.372, 79.528, 75.593, 74.727]

Loading data from ../data/retacred/test_bottom.json with batch size 50...
41 batches created for ../data/retacred/test_bottom.json


100%|██████████| 41/41 [00:00<00:00, 65.92it/s]


Precision (micro): 76.505%
   Recall (micro): 82.976%
       F1 (micro): 79.610%
Evaluation ended.
